In [ ]:
!git clone https://github.com/piotrszczypior/backdoor-resnet.git

In [ ]:
import sys
import os

notebook_dir = os.path.abspath(".")
project_path = os.path.join(notebook_dir, "backdoor-resnet")
sys.path.append(project_path)

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from src.train import training_loop
from src.dataset import BackdooredDataset
from src.model import get_resnet_model
from src.backdoor import gaussian_noise_static_trigger


print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Config:
    BATCH_SIZE = 128
    WEIGHT_DECAY = 0.0001
    EPOCH_NUMBER = 200
    MOMENTUM = 0.9
    INITIAL_LEARNING_RATE = 0.1


def get_model():
    model = get_resnet_model(100)
    model.to(DEVICE)

    return model


def get_data_loaders():
    transform_train = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761]
            ),
        ]
    )

    transform_test = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761]
            ),
        ]
    )

    train_dataset = BackdooredDataset(
        dataset="CIFAR100",
        train=True,
        transform=transform_train,
        backdoor=True,
        mode="append",
        label_mode="label_flip",
        trigger_fn=gaussian_noise_static_trigger,
        label_flip_target=66,  # raccoon
        p=0.05,
    )

    test_dataset = BackdooredDataset(
        dataset="CIFAR100", train=False, transform=transform_test, backdoor=False
    )

    train_dataloader = DataLoader(
        train_dataset, Config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
    )

    test_dataloader = DataLoader(
        test_dataset, Config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
    )

    return train_dataloader, test_dataloader


def train():
    model = get_model()
    train_data_loader, test_data_loader = get_data_loaders()

    training_loop(model, Config, train_data_loader, test_data_loader)

In [ ]:
train()